## import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LinearRegression

In [ ]:
df=pd.read_csv("D:\\Iris.csv")

In [ ]:
df.head()

In [ ]:
df.drop(["Id"], axis=1, inplace=True)

In [ ]:
df

In [ ]:
df["category"] = pd.qcut(df["SepalLengthCm"], q=3, labels=["1","2","3"])
df.tail()

In [ ]:
sp = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

for train_idx, test_idx in sp.split(df,df["category"]):

    train_df = df.loc[train_idx]
    test_df = df.loc[test_idx]

In [ ]:
df["category"]

In [ ]:
X_train = train_df[["SepalWidthCm", "PetalLengthCm", "PetalWidthCm"]]
y_train = train_df["Species"]

X_test = test_df[["SepalWidthCm", "PetalLengthCm", "PetalWidthCm"]]
y_test = test_df["Species"]

In [ ]:
y_train = pd.get_dummies(y_train)
y_test = pd.get_dummies(y_test)

In [ ]:
X_train, X_test = X_train.align(X_test, join="left", axis=1, fill_value=0)

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
model = LogisticRegression()

In [ ]:
model.fit(X_train, y_train.idxmax(axis=1)) # idxmax():- maximum value wale column ka naam return karta hai.


In [ ]:
pred = model.predict(X_test)

In [ ]:
pred

In [2]:
#PIPELINE
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.impute import SimpleImputer

In [3]:
df=pd.read_csv("D:\\Iris.csv")
df.head()

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa


In [4]:
df.drop(columns=["Id"],axis=1,inplace= True)

In [5]:
df.head()

,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


In [6]:
x=df.drop(columns=["Species"],axis=1).copy()
y=df["Species"]

In [7]:
x_train , x_test , y_train , y_test= train_test_split(x,y, test_size=0.2, random_state= 42)

In [8]:
num_pipe=Pipeline([("impute", SimpleImputer(strategy= "mean")),("scaler", StandardScaler())])
cat_pipe=Pipeline([("impute", SimpleImputer(strategy= "most_frequent")), ("encoder", OneHotEncoder(sparse_output=False))])

In [13]:
complete_pipe=ColumnTransformer([("number",num_pipe, x_train.select_dtypes(include = np.number).columns.tolist()),("category", cat_pipe, x_train.select_dtypes(exclude= np.number).columns.tolist())],verbose_feature_names_out=False, verbose=True).set_output(transform="pandas")

In [16]:
complete_pipe.fit_transform(x_train)

[ColumnTransformer] ........ (1 of 1) Processing number, total=   0.0s


,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm
22,-1.473937,1.220379,-1.563987,-1.309484
15,-0.133071,3.020017,-1.277280,-1.042922
65,1.085898,0.095606,0.385621,0.289886
11,-1.230143,0.770470,-1.219939,-1.309484
42,-1.717731,0.320560,-1.391963,-1.309484
...,...,...,...,...
71,0.354517,-0.579258,0.156255,0.156605
106,-1.108246,-1.254122,0.442962,0.689728
14,-0.011174,2.120198,-1.449304,-1.309484
92,-0.011174,-1.029168,0.156255,0.023324


In [22]:
final_pipe=([("complete", complete_pipe), ("model",  LogisticRegression())])

In [23]:
scores = cross_val_score(
    final_pipe,
    x_train,
    y_train,
    cv=StratifiedKFold,
    scoring="neg_root_mean_squared_error"
)

InvalidParameterError: The 'estimator' parameter of cross_val_score must be an object implementing 'fit'. Got [('complete', ColumnTransformer(transformers=[('number',
                                 Pipeline(steps=[('impute', SimpleImputer()),
                                                 ('scaler', StandardScaler())]),
                                 ['SepalLengthCm', 'SepalWidthCm',
                                  'PetalLengthCm', 'PetalWidthCm']),
                                ('category',
                                 Pipeline(steps=[('impute',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(sparse_output=False))]),
                                 [])],
                  verbose=True, verbose_feature_names_out=False)), ('model', LogisticRegression())] instead.